In [1]:
# !pip install ISLP

In [2]:
import pandas as pd
from datetime import datetime

from pandas import DataFrame

In [3]:
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [4]:
import statsmodels.api as sm
# from ISLP.utils import load_data      # error
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

from sklearn.model_selection import train_test_split

from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

## Read training and test datasets

In [5]:
# helper function for reading datatset
def read_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d') # convert it to datatime

    return df

In [6]:
df_train = read_data('data/kospi_train.csv')
df_test = read_data('data/kospi_test.csv')

len(df_train), len(df_test)

(986, 244)

In [7]:
# the training dataset has daily KOSPI index from 2019 to 2022
df_train

,Date,Open,Low,High,Close,Volume
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800
...,...,...,...,...,...,...
981,2022-12-23,2325.860107,2311.899902,2333.080078,2313.689941,367000
982,2022-12-26,2312.540039,2304.199951,2321.919922,2317.139893,427600
983,2022-12-27,2327.520020,2321.479980,2335.989990,2332.790039,448300
984,2022-12-28,2296.449951,2276.899902,2296.449951,2280.449951,405700


In [8]:
# the test dataset has daily KOSPI index in 2023
df_test

,Date,Open,Low,High,Close,Volume
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300
...,...,...,...,...,...,...
239,2023-12-21,2598.370117,2587.159912,2610.810059,2600.020020,578300
240,2023-12-22,2617.719971,2599.510010,2621.370117,2599.510010,466000
241,2023-12-26,2609.439941,2594.649902,2612.139893,2602.590088,439500
242,2023-12-27,2599.350098,2590.080078,2613.500000,2613.500000,349700


In [9]:
# a function for residual plot!
# use sns.regplot for fancier plot
def plot_residue(pred, resid):
    """
    inputs: 
        pred - predicted values
        resid - residuals
    """
    
    import seaborn as sns

    res=sns.regplot(x=pred, y=resid, lowess=True, 
            line_kws={'color':'r', 'lw':1},
            scatter_kws={'facecolors':'None', 'edgecolors':'k', 'alpha':0.5})
    XLIM=res.axes.xaxis.get_data_interval()
    #res.axes.hlines(0,XLIM[0], XLIM[1], linestyles='dotted')
    plt.hlines(0,XLIM[0], XLIM[1], linestyles='dotted')
    plt.xlabel('fitted values')
    plt.ylabel('residuals')
    plt.title('Residuals vs. fitted')

## Part 1. Train regression models to predict the next day's `close` using `Open`, `Low`, `High`, `Close`, `Volume` of previous days as predictors using *only* df_train. Cross-validate to select the best model. Evaluate the accuracy of your model using `df_test`.


In [10]:
# for linear regression, preprocessing is needed
# the column means [day 1, day 2, day 3, day 4, day 5]
# the answer is Close of day 6
# to prevent IndexError, the last n-1 columns are trimmed
def get_prev_days_df(df: DataFrame, n_days=3) -> DataFrame: 
    '''make a row using previous n days' variables and the next day's Close
    '''
    df_new = df.copy()

    for j in range(1,n_days):
        for column in ['Open', 'Low', 'High', 'Close', 'Volume']:
            column_name = f'{j}_{column}'
            df_new[column_name] = 0
            for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
    df_new['Answer'] = 0
    for i in range(df_new.shape[0]-n_days): df_new.loc[i, 'Answer'] = df.loc[i+n_days, 'Close']

    df_new = df_new[df_new['Answer'] > 0]

    return df_new

In [11]:
df_train_new = get_prev_days_df(df_train, n_days=5)
df_train_new

C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2011.81005859375' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1991.6500244140625' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2014.719970703125' has dtype incom

,Date,Open,Low,High,Close,Volume,1_Open,1_Low,1_High,1_Close,...,3_Low,3_High,3_Close,3_Volume,4_Open,4_Low,4_High,4_Close,4_Volume,Answer
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400,2011.810059,1991.650024,2014.719971,1993.699951,...,2030.900024,2048.060059,2037.099976,440200,2038.680054,2023.589966,2042.699951,2025.270020,397800,2064.709961
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000,1992.400024,1984.530029,2011.560059,2010.250000,...,2023.589966,2042.699951,2025.270020,397800,2034.189941,2034.189941,2068.229980,2064.709961,386200,2063.280029
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000,2034.239990,2030.900024,2048.060059,2037.099976,...,2034.189941,2068.229980,2064.709961,386200,2065.729980,2057.159912,2072.810059,2063.280029,382900,2075.570068
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200,2038.680054,2023.589966,2042.699951,2025.270020,...,2057.159912,2072.810059,2063.280029,382900,2070.360107,2063.989990,2076.989990,2075.570068,380100,2064.520020
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800,2034.189941,2034.189941,2068.229980,2064.709961,...,2063.989990,2076.989990,2075.570068,380100,2070.489990,2059.459961,2073.939941,2064.520020,432900,2097.179932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
976,2022-12-16,2329.750000,2326.830078,2360.439941,2360.020020,414200,2350.780029,2342.280029,2358.760010,2352.169922,...,2325.780029,2347.000000,2328.949951,329400,2340.000000,2335.750000,2356.729980,2356.729980,552800,2313.689941
977,2022-12-19,2350.780029,2342.280029,2358.760010,2352.169922,323600,2344.729980,2324.659912,2353.860107,2333.290039,...,2335.750000,2356.729980,2356.729980,552800,2325.860107,2311.899902,2333.080078,2313.689941,367000,2317.139893
978,2022-12-20,2344.729980,2324.659912,2353.860107,2333.290039,358100,2346.389893,2325.780029,2347.000000,2328.949951,...,2311.899902,2333.080078,2313.689941,367000,2312.540039,2304.199951,2321.919922,2317.139893,427600,2332.790039
979,2022-12-21,2346.389893,2325.780029,2347.000000,2328.949951,329400,2340.000000,2335.750000,2356.729980,2356.729980,...,2304.199951,2321.919922,2317.139893,427600,2327.520020,2321.479980,2335.989990,2332.790039,448300,2280.449951


In [12]:
X, Y = df_train_new.drop(columns=['Date','Answer']), df_train_new['Answer']
column_name = list(X.columns)

X = MS(list(X.columns)).fit_transform(df_train_new)
Y = df_train_new['Answer']
M_lm = sm.OLS(Y, X).fit()
summarize(M_lm)

,coef,std err,t,P>|t|
intercept,8.177100,5.539000,1.476,0.140
Open,0.205300,0.099000,2.066,0.039
Low,-0.149800,0.110000,-1.356,0.175
High,-0.296800,0.122000,-2.430,0.015
Close,0.088500,0.110000,0.801,0.423
Volume,-0.000009,0.000005,-1.792,0.074
1_Open,0.281900,0.110000,2.557,0.011
1_Low,-0.060600,0.116000,-0.522,0.602
1_High,-0.116900,0.126000,-0.929,0.353
1_Close,-0.071100,0.112000,-0.636,0.525


In [ ]:
# (P > |t|) < 0.05
# Open, High, 1_Open, 1_Volume, 2_High, 2_Close, 4_High, 4_Close
# Open, Close, and High are better?

In [14]:
X, Y = df_train_new.drop(columns=['Date','Answer']), df_train_new['Answer']
column_name = list(X.columns)

cv_error = np.zeros(5)
M = sklearn_sm(sm.OLS)
cv = cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0)

# 1: use all columns

M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
cv_error[0] = np.mean(M_CV['test_score'])

# 2: use only previous 3 days
tmp_column = list(filter(lambda x: x.startswith(('2', '3', '4')), column_name))

X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
cv_error[1] = np.mean(M_CV['test_score'])

# 3: use Open, Close, High
tmp_column = list(filter(lambda x: x.endswith(('Open', 'Close', 'High')), column_name))

X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
cv_error[2] = np.mean(M_CV['test_score'])

# 4: use Open, Close, High with 3 days
tmp_column = list(filter(lambda x: x.endswith(('Close', 'High')), column_name))

X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
cv_error[3] = np.mean(M_CV['test_score'])

# 4: use Close, High with 3 days
tmp_column = list(filter(lambda x: x.endswith(('Close', 'High')) and x.startswith(('2','3','4')), column_name))

X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
cv_error[4] = np.mean(M_CV['test_score'])

cv_error

array([885.9519963 , 874.93737737, 858.64457453, 849.47377444,
       848.11117984])

In [15]:
# get fifth method's model

tmp_column = list(filter(lambda x: x.endswith(('Close', 'High')) and x.startswith(('2','3','4')), column_name))
X = df_train_new[tmp_column]
column_mm = MS(tmp_column)
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results = model.fit()


In [16]:
# get test dataset
df_test_new = get_prev_days_df(df_test, n_days=5)
df_test_new

C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2230.97998046875' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2180.669921875' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12020\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2230.97998046875' has dtype incompatib

,Date,Open,Low,High,Close,Volume,1_Open,1_Low,1_High,1_Close,...,3_Low,3_High,3_Close,3_Volume,4_Open,4_Low,4_High,4_Close,4_Volume,Answer
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100,2230.979980,2180.669922,2230.979980,2218.679932,...,2252.969971,2281.389893,2264.649902,430800,2253.399902,2253.270020,2300.620117,2289.969971,398300,2350.189941
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000,2205.979980,2198.820068,2260.060059,2255.979980,...,2253.270020,2300.620117,2289.969971,398300,2315.870117,2312.560059,2351.060059,2350.189941,341100,2351.310059
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700,2268.199951,2252.969971,2281.389893,2264.649902,...,2312.560059,2351.060059,2350.189941,341100,2348.040039,2344.179932,2370.179932,2351.310059,359600,2359.530029
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800,2253.399902,2253.270020,2300.620117,2289.969971,...,2344.179932,2370.179932,2351.310059,359600,2364.050049,2350.360107,2369.659912,2359.530029,368800,2365.100098
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300,2315.870117,2312.560059,2351.060059,2350.189941,...,2350.360107,2369.659912,2359.530029,368800,2376.719971,2358.330078,2377.800049,2365.100098,580100,2386.090088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,2023-12-14,2547.739990,2532.159912,2549.649902,2544.179932,530100,2558.439941,2555.300049,2574.229980,2563.560059,...,2556.520020,2570.060059,2568.550049,392500,2586.989990,2584.850098,2615.379883,2614.300049,570400,2600.020020
235,2023-12-15,2558.439941,2555.300049,2574.229980,2563.560059,465300,2568.770020,2556.050049,2573.129883,2566.860107,...,2584.850098,2615.379883,2614.300049,570400,2598.370117,2587.159912,2610.810059,2600.020020,578300,2599.510010
236,2023-12-18,2568.770020,2556.050049,2573.129883,2566.860107,383000,2564.810059,2556.520020,2570.060059,2568.550049,...,2587.159912,2610.810059,2600.020020,578300,2617.719971,2599.510010,2621.370117,2599.510010,466000,2602.590088
237,2023-12-19,2564.810059,2556.520020,2570.060059,2568.550049,392500,2586.989990,2584.850098,2615.379883,2614.300049,...,2599.510010,2621.370117,2599.510010,466000,2609.439941,2594.649902,2612.139893,2602.590088,439500,2613.500000


In [17]:
# get test MSE

tmp_column_test = list(filter(lambda x: x.endswith(('Close', 'High')) and x.startswith(('2','3','4')), column_name))
X = df_test_new[tmp_column_test]
X_test = column_mm.transform(X)
y_test = Y
valid_pred = results.predict(X_test)
np.mean((y_test - valid_pred)**2)

171887.02642139088

## Part 2. Extend the regression model by adding some extra features of your choice. You can use any statistics publicly available. 

In [18]:
# many stock dataset includes increased/decreased price of each day
# as final method is the best, just use Open, Close change in 3 days
def get_dif_df(df_new: DataFrame) -> DataFrame:
    tmp_column = list(filter(lambda x: x.endswith(('Close', 'High')), df_new.columns))
    df_dif = df_new[tmp_column].copy()
    df_dif['21_Close_%_dif'] = (df_dif['2_Close'] - df_dif['1_Close']) / df_dif['1_Close'] * 100
    df_dif['32_Close_%_dif'] = (df_dif['3_Close'] - df_dif['2_Close']) / df_dif['2_Close'] * 100
    df_dif['43_Close_%_dif'] = (df_dif['4_Close'] - df_dif['3_Close']) / df_dif['3_Close'] * 100

    df_dif['21_High_%_dif'] = (df_dif['2_High'] - df_dif['1_High']) / df_dif['1_High'] * 100
    df_dif['32_High_%_dif'] = (df_dif['3_High'] - df_dif['2_High']) / df_dif['2_High'] * 100
    df_dif['43_High_%_dif'] = (df_dif['4_High'] - df_dif['3_High']) / df_dif['3_High'] * 100

    tmp_column = list(filter(lambda x: x.endswith('dif') or x.startswith('2'), df_dif.columns))
    df_dif = df_dif[tmp_column].copy()

    return df_dif

In [19]:
df_train_dif = get_dif_df(df_train_new)
df_train_dif

,2_High,2_Close,21_Close_%_dif,32_Close_%_dif,43_Close_%_dif,21_High_%_dif,32_High_%_dif,43_High_%_dif
0,2011.560059,2010.250000,0.830117,1.335654,-0.580725,-0.156841,1.814512,-0.261716
1,2048.060059,2037.099976,1.335654,-0.580725,1.947392,1.814512,-0.261716,1.249818
2,2042.699951,2025.270020,-0.580725,1.947392,-0.069256,-0.261716,1.249818,0.221449
3,2068.229980,2064.709961,1.947392,-0.069256,0.595655,1.249818,0.221449,0.201655
4,2072.810059,2063.280029,-0.069256,0.595655,-0.532386,0.221449,0.201655,-0.146849
...,...,...,...,...,...,...,...,...
976,2353.860107,2333.290039,-0.802658,-0.186007,1.192813,-0.207732,-0.291441,0.414571
977,2347.000000,2328.949951,-0.186007,1.192813,-1.826261,-0.291441,0.414571,-1.003505
978,2356.729980,2356.729980,1.192813,-1.826261,0.149110,0.414571,-1.003505,-0.478344
979,2333.080078,2313.689941,-1.826261,0.149110,0.675408,-1.003505,-0.478344,0.605967


In [20]:
# get dif model

X = df_train_dif
column_mm = MS(df_train_dif)
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results_dif = model.fit()
results_dif.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Answer   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.995
Method:                 Least Squares   F-statistic:                 2.616e+04
Date:                Thu, 01 May 2025   Prob (F-statistic):               0.00
Time:                        15:11:21   Log-Likelihood:                -4703.3
No. Observations:                 981   AIC:                             9425.
Df Residuals:                     972   BIC:                             9469.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
intercept          3.9346      5.538      0.710      0.478      -6.933      14.802
2_High             0.4450      0.098      4.554      0.000       0.253       0.637
2_Close            0.5505      0.098      5.607      0.000       0.358       0.743
21_Close_%_dif     3.1686      1.506      2.104      0.036       0.214       6.124
32_Close_%_dif    19.2846      1.965      9.813      0.000      15.428      23.141
43_Close_%_dif    18.5138      1.324     13.987      0.000      15.916      21.111
21_High_%_dif     -1.5598      1.407     -1.109      0.268      -4.321       1.201
32_High_%_dif      5.2141      2.372      2.198      0.028       0.560       9.869
43_High_%_dif      7.4256      1.905      3.899      0.000       3.688      11.163
==============================================================================
Omnibus:                       83.482   Durbin-Watson:                   1.837
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              310.247
Skew:                          -0.329   Prob(JB):                     4.27e-68
Kurtosis:                       5.675   Cond. No.                     2.12e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.12e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [21]:
correlation_matrix = X.iloc[:, 1:].corr()
correlation_matrix

,2_Close,21_Close_%_dif,32_Close_%_dif,43_Close_%_dif,21_High_%_dif,32_High_%_dif,43_High_%_dif
2_Close,1.000000,0.019591,-0.045323,-0.043370,0.023316,-0.020832,-0.047109
21_Close_%_dif,0.019591,1.000000,-0.010404,0.119120,0.665010,0.424950,0.135862
32_Close_%_dif,-0.045323,-0.010404,1.000000,-0.012048,0.103324,0.665383,0.422939
43_Close_%_dif,-0.043370,0.119120,-0.012048,1.000000,0.051132,0.102294,0.665991
21_High_%_dif,0.023316,0.665010,0.103324,0.051132,1.000000,0.215275,0.087230
32_High_%_dif,-0.020832,0.424950,0.665383,0.102294,0.215275,1.000000,0.214565
43_High_%_dif,-0.047109,0.135862,0.422939,0.665991,0.087230,0.214565,1.000000


In [22]:
df_test_dif = get_dif_df(df_test_new)
df_test_dif

,2_High,2_Close,21_Close_%_dif,32_Close_%_dif,43_Close_%_dif,21_High_%_dif,32_High_%_dif,43_High_%_dif
0,2260.060059,2255.979980,1.681182,0.384308,1.118057,1.303467,0.943773,0.842917
1,2281.389893,2264.649902,0.384308,1.118057,2.629728,0.943773,0.842917,2.192450
2,2300.620117,2289.969971,1.118057,2.629728,0.047661,0.842917,2.192450,0.813245
3,2351.060059,2350.189941,2.629728,0.047661,0.349591,2.192450,0.813245,-0.021940
4,2370.179932,2351.310059,0.047661,0.349591,0.236067,0.813245,-0.021940,0.343515
...,...,...,...,...,...,...,...,...
234,2573.129883,2566.860107,0.128729,0.065837,1.781161,-0.042735,-0.119303,1.763376
235,2570.060059,2568.550049,0.065837,1.781161,-0.546228,-0.119303,1.763376,-0.174729
236,2615.379883,2614.300049,1.781161,-0.546228,-0.019616,1.763376,-0.174729,0.404474
237,2610.810059,2600.020020,-0.546228,-0.019616,0.118487,-0.174729,0.404474,-0.352115


In [23]:
# get final test MSE

X = df_test_dif
X_test = column_mm.transform(X)
y_test = Y
test_pred = results_dif.predict(X_test)
np.mean((y_test - test_pred)**2)

171276.9322089324

In [24]:
# remove high-coefficient 'High'
close_only_column = list(filter(lambda x: x.find(('Close')) != -1, df_train_dif.columns))

X = df_train_dif[close_only_column]
column_mm = MS(df_train_dif[close_only_column])
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results_dif = model.fit()
results_dif.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Answer   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.995
Method:                 Least Squares   F-statistic:                 5.080e+04
Date:                Thu, 01 May 2025   Prob (F-statistic):               0.00
Time:                        15:11:21   Log-Likelihood:                -4719.6
No. Observations:                 981   AIC:                             9449.
Df Residuals:                     976   BIC:                             9474.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
intercept          5.9191      5.601      1.057      0.291      -5.073      16.911
2_Close            0.9975      0.002    450.366      0.000       0.993       1.002
21_Close_%_dif     0.2537      0.767      0.331      0.741      -1.252       1.759
32_Close_%_dif    24.9651      0.762     32.749      0.000      23.469      26.461
43_Close_%_dif    22.9467      0.767     29.920      0.000      21.442      24.452
==============================================================================
Omnibus:                       91.391   Durbin-Watson:                   1.869
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              579.488
Skew:                          -0.004   Prob(JB):                    1.47e-126
Kurtosis:                       6.765   Cond. No.                     1.49e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.49e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [25]:
# get final test MSE

X = df_test_dif[close_only_column]
X_test = column_mm.transform(X)
y_test = Y
test_pred = results_dif.predict(X_test)
np.mean((y_test - test_pred)**2)

172449.39903082149

In [26]:
# remove high-coefficient 'Close'
close_only_column = list(filter(lambda x: x.find(('High')) != -1, df_train_dif.columns))

X = df_train_dif[close_only_column]
column_mm = MS(df_train_dif[close_only_column])
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results_dif = model.fit()
results_dif.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Answer   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.994
Method:                 Least Squares   F-statistic:                 4.363e+04
Date:                Thu, 01 May 2025   Prob (F-statistic):               0.00
Time:                        15:11:21   Log-Likelihood:                -4793.8
No. Observations:                 981   AIC:                             9598.
Df Residuals:                     976   BIC:                             9622.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
intercept         4.3608      6.049      0.721      0.471      -7.510      16.232
2_High            0.9920      0.002    417.265      0.000       0.987       0.997
21_High_%_dif     1.0242      1.060      0.966      0.334      -1.055       3.104
32_High_%_dif    24.8745      1.081     23.009      0.000      22.753      26.996
43_High_%_dif    29.5582      1.060     27.881      0.000      27.478      31.639
==============================================================================
Omnibus:                      131.191   Durbin-Watson:                   1.466
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              471.172
Skew:                          -0.610   Prob(JB):                    4.86e-103
Kurtosis:                       6.168   Cond. No.                     1.50e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.5e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [27]:
# get final test MSE

X = df_test_dif[close_only_column]
X_test = column_mm.transform(X)
y_test = Y
test_pred = results_dif.predict(X_test)
np.mean((y_test - test_pred)**2)

169807.39403346644